# Experiment 9E: Full Fine-Tuning (No LoRA)

Eliminate LoRA capacity / fused-QKV asymmetry entirely.
Train both BERT-base and ModernBERT-base with ALL parameters unfrozen.
If ModernBERT catches up, the gap was a LoRA artifact.
If BERT still wins, the gap is from pre-training.

In [ ]:
%%capture
!pip install -q "datasets>=3.4.1,<4.0.0" scikit-learn matplotlib seaborn accelerate transformers sentencepiece protobuf

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, training_args
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import json
import gc
import matplotlib.pyplot as plt
import seaborn as sns

NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
FPB_SOURCE = 5

label_dict = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}

ds = load_dataset("neoyipeng/financial_reasoning_aggregated")
ds = ds.filter(lambda x: x["task"] == "sentiment")
ds = ds.filter(lambda x: x["source"] != FPB_SOURCE)

remove_cols = [c for c in ds["train"].column_names if c not in ("text", "labels")]
ds = ds.map(
    lambda ex: {
        "text": ex["text"],
        "labels": np.eye(NUM_CLASSES)[label_dict[ex["label"]]],
    },
    remove_columns=remove_cols,
)

print(f"Train: {len(ds['train']):,}  |  Val: {len(ds['validation']):,}  |  Test: {len(ds['test']):,}")

def load_fpb_file(agree_level):
    from huggingface_hub import hf_hub_download
    import zipfile, os
    path = hf_hub_download("financial_phrasebank", "data/FinancialPhraseBank-v1.0.zip", repo_type="dataset")
    extract_dir = "/tmp/fpb_data"
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(path, "r") as z:
        z.extractall(extract_dir)
    filename = {"50agree": "Sentences_50Agree.txt", "allagree": "Sentences_AllAgree.txt"}[agree_level]
    filepath = os.path.join(extract_dir, "FinancialPhraseBank-v1.0", filename)
    sentences, labels = [], []
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            at_idx = line.rfind("@")
            if at_idx == -1:
                continue
            label_str = line[at_idx+1:].strip().lower()
            if label_str in label_map:
                sentences.append(line[:at_idx].strip())
                labels.append(label_map[label_str])
    return {"sentence": sentences, "label": labels}

def load_fpb_dataset(agree_level="50agree"):
    try:
        config = f"sentences_{agree_level}"
        return load_dataset("financial_phrasebank", config, trust_remote_code=True)["train"]
    except Exception:
        return load_fpb_file(agree_level)

fpb_50 = load_fpb_dataset("50agree")
fpb_all = load_fpb_dataset("allagree")
print(f"FPB 50agree: {len(fpb_50['sentence']):,}  |  FPB allAgree: {len(fpb_all['sentence']):,}")

In [ ]:
label_dict = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}

ds = load_dataset("neoyipeng/financial_reasoning_aggregated")
ds = ds.filter(lambda x: x["task"] == "sentiment")
ds = ds.filter(lambda x: x["source"] != FPB_SOURCE)

remove_cols = [c for c in ds["train"].column_names if c not in ("text", "labels")]
ds = ds.map(
    lambda ex: {
        "text": ex["text"],
        "labels": np.eye(NUM_CLASSES)[label_dict[ex["label"]]],
    },
    remove_columns=remove_cols,
)

print(f"Train: {len(ds['train']):,}  |  Val: {len(ds['validation']):,}  |  Test: {len(ds['test']):,}")

fpb_50 = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
fpb_all = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]
print(f"FPB 50agree: {len(fpb_50):,}  |  FPB allAgree: {len(fpb_all):,}")

## 2. Evaluation Helper

In [ ]:
def evaluate_on_fpb(model, tokenizer, fpb_dataset, batch_size=32):
    fpb_texts = fpb_dataset["sentence"]
    fpb_labels = fpb_dataset["label"]
    all_preds = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(fpb_texts), batch_size), desc="Evaluating"):
            batch = fpb_texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs = {k: v.cuda() for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
    y_true = np.array(fpb_labels)
    y_pred = np.array(all_preds)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    report = classification_report(y_true, y_pred, target_names=LABEL_NAMES)
    cm = confusion_matrix(y_true, y_pred)
    print(f"Accuracy: {acc:.4f} ({int(acc * len(y_true))}/{len(y_true)})")
    print(f"Macro F1: {macro_f1:.4f}")
    print(report)
    return {"accuracy": acc, "macro_f1": macro_f1, "cm": cm}

## 3. Full Fine-Tune BERT-base

In [ ]:
print("Full fine-tuning BERT-base-uncased (ALL parameters)...")
bert_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=NUM_CLASSES)
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model.gradient_checkpointing_enable()

total_params = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

def tokenize_bert(examples):
    return bert_tokenizer(examples["text"], truncation=True, max_length=512)

train_data = ds["train"].map(tokenize_bert, batched=True)
val_data = ds["validation"].map(tokenize_bert, batched=True)

bert_trainer = Trainer(
    model=bert_model,
    processing_class=bert_tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=TrainingArguments(
        output_dir="trainer_output_09e_bert_full",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        warmup_ratio=0.1,
        fp16=True,
        optim=training_args.OptimizerNames.ADAMW_TORCH,
        learning_rate=2e-5,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        num_train_epochs=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="epoch",
        gradient_checkpointing=True,
        report_to="none",
        save_total_limit=2,
    ),
    compute_metrics=lambda eval_pred: {
        "accuracy": accuracy_score(
            eval_pred[1].argmax(axis=-1), eval_pred[0].argmax(axis=-1)
        )
    },
)

bert_trainer.train()
bert_model = bert_model.cuda().eval()

In [ ]:
print("\n" + "=" * 60)
print("BERT-base FULL FT — FPB sentences_50agree")
print("=" * 60)
bert_full_50 = evaluate_on_fpb(bert_model, bert_tokenizer, fpb_50)

print("\n" + "=" * 60)
print("BERT-base FULL FT — FPB sentences_allAgree")
print("=" * 60)
bert_full_all = evaluate_on_fpb(bert_model, bert_tokenizer, fpb_all)

del bert_model, bert_trainer
gc.collect()
torch.cuda.empty_cache()

## 4. Full Fine-Tune ModernBERT-base

In [ ]:
print("Full fine-tuning ModernBERT-base (ALL parameters)...")
mb_model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base", num_labels=NUM_CLASSES)
mb_tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
mb_model.gradient_checkpointing_enable()

total_params = sum(p.numel() for p in mb_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

def tokenize_mb(examples):
    return mb_tokenizer(examples["text"], truncation=True, max_length=512)

train_data_mb = ds["train"].map(tokenize_mb, batched=True)
val_data_mb = ds["validation"].map(tokenize_mb, batched=True)

mb_trainer = Trainer(
    model=mb_model,
    processing_class=mb_tokenizer,
    train_dataset=train_data_mb,
    eval_dataset=val_data_mb,
    args=TrainingArguments(
        output_dir="trainer_output_09e_mb_full",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        warmup_ratio=0.1,
        fp16=True,
        optim=training_args.OptimizerNames.ADAMW_TORCH,
        learning_rate=2e-5,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        num_train_epochs=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="epoch",
        gradient_checkpointing=True,
        report_to="none",
        save_total_limit=2,
    ),
    compute_metrics=lambda eval_pred: {
        "accuracy": accuracy_score(
            eval_pred[1].argmax(axis=-1), eval_pred[0].argmax(axis=-1)
        )
    },
)

mb_trainer.train()
mb_model = mb_model.cuda().eval()

In [ ]:
print("\n" + "=" * 60)
print("ModernBERT-base FULL FT — FPB sentences_50agree")
print("=" * 60)
mb_full_50 = evaluate_on_fpb(mb_model, mb_tokenizer, fpb_50)

print("\n" + "=" * 60)
print("ModernBERT-base FULL FT — FPB sentences_allAgree")
print("=" * 60)
mb_full_all = evaluate_on_fpb(mb_model, mb_tokenizer, fpb_all)

del mb_model, mb_trainer
gc.collect()
torch.cuda.empty_cache()

## 5. Comparison: LoRA vs Full Fine-Tuning

In [ ]:
print("\nLoRA vs Full Fine-Tuning Comparison")
print("=" * 80)
print(f"{'Config':<35} {'Params':>12} {'FPB 50agree':>12} {'FPB allAgree':>12}")
print("-" * 80)
print(f"{'BERT + LoRA r=16 (NB05)':<35} {'~0.89M':>12} {'95.19%':>12} {'99.16%':>12}")
print(f"{'BERT + Full FT (this exp)':<35} {'~110M':>12} {bert_full_50['accuracy']:>11.2%} {bert_full_all['accuracy']:>12.2%}")
print(f"{'ModernBERT + LoRA r=16 (NB01)':<35} {'~1.1M':>12} {'91.19%':>12} {'99.03%':>12}")
print(f"{'ModernBERT + LoRA r=48 (NB08)':<35} {'~3.3M':>12} {'82.29%':>12} {'94.30%':>12}")
print(f"{'ModernBERT + Full FT (this exp)':<35} {'~149M':>12} {mb_full_50['accuracy']:>11.2%} {mb_full_all['accuracy']:>12.2%}")
print()
gap = (bert_full_50['accuracy'] - mb_full_50['accuracy']) * 100
print(f"Full FT gap (BERT - ModernBERT): {gap:+.2f}pp on 50agree")
if abs(gap) < 1.5:
    print("-> Gap CLOSED under full FT. LoRA asymmetry was the main cause.")
elif gap > 2:
    print("-> Gap PERSISTS under full FT. Pre-training alignment is the cause.")
else:
    print("-> Gap partially reduced. Both LoRA asymmetry and pre-training contribute.")

results_09e = {
    "bert_full_50agree": bert_full_50["accuracy"],
    "bert_full_allagree": bert_full_all["accuracy"],
    "bert_full_50_f1": bert_full_50["macro_f1"],
    "mb_full_50agree": mb_full_50["accuracy"],
    "mb_full_allagree": mb_full_all["accuracy"],
    "mb_full_50_f1": mb_full_50["macro_f1"],
}
with open("results_09e.json", "w") as f:
    json.dump(results_09e, f, indent=2)
print("\nSaved to results_09e.json")

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (res, title) in zip(axes, [
    (bert_full_50, "BERT-base Full FT"),
    (mb_full_50, "ModernBERT Full FT"),
]):
    sns.heatmap(
        res["cm"], annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax,
    )
    ax.set_title(f"{title}\nAcc={res['accuracy']:.2%}  F1={res['macro_f1']:.2%}")
    ax.set_ylabel("True")
    ax.set_xlabel("Predicted")

plt.suptitle("Exp 9E: Full Fine-Tuning — FPB sentences_50agree", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("09e_full_finetune.png", dpi=150, bbox_inches="tight")
plt.show()